In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q streamlit plotly pyngrok

In [ ]:
import os

base="/content/drive/MyDrive/Hackathon/var_control_cockpit"

print(os.listdir(base))
print(os.listdir(f"{base}/data/processed"))
print(os.listdir(f"{base}/reports"))

['data', 'outputs', 'audit', 'reports', 'models']
['rule_observation_alerts.csv', 'enriched_market_data.csv', 'rule_portfolio_alerts.csv', 'market_data_with_rules.csv', 'market_data_with_ai.csv', 'market_data_with_context.csv', 'context_observations.csv', 'market_data_prioritised.csv', 'investigation_pack.csv']
['block4_rule_quality_checks.csv', 'block4_rule_summary.csv', 'gpt_investigation_reports.csv']


In [ ]:
!streamlit run app.py >/content/log.txt 2>&1 &

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("2ykyRPU0jxUoqpoiCFqt6f8fiuE_49jt3Nb9uUnD3v4GBMU2F")

public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://4be7-35-199-59-109.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import os
from datetime import datetime

# -------------------------------------------------
# CONFIG
# -------------------------------------------------

st.set_page_config(
    page_title="AI Market Risk Control Cockpit",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)

DATA_PATH="/content/drive/MyDrive/Hackathon/var_control_cockpit"

# -------------------------------------------------

# CSS

# -------------------------------------------------

st.markdown("""

<style>



.main{

    background-color:#0E1117;

}

/* ---------- Sidebar ---------- */

section[data-testid="stSidebar"]{

    background:#17212B;

}

/* ---------- Metric Cards ---------- */

.metric-box{

    background:#243447;

    border-left:6px solid #00C8FF;

    padding:18px;

    border-radius:12px;

    color:#FFFFFF;

    box-shadow:0 2px 8px rgba(0,0,0,0.25);

    margin-bottom:12px;

}

/* ---------- Headers ---------- */

h1,h2,h3,h4{

    color:#FFFFFF;

}



/* ---------- Tables ---------- */

thead tr th{

    background:#1E293B !important;

    color:white !important;

}

tbody tr{

    background:#243447 !important;

    color:white !important;

}

/* ---------- Status ---------- */

.live{

    color:#22C55E;

    font-weight:bold;

}

.red{

    color:#EF4444;

    font-size:30px;

    font-weight:bold;

}

.orange{

    color:#F59E0B;

    font-size:30px;

    font-weight:bold;

}

.green{

    color:#22C55E;

    font-size:30px;

    font-weight:bold;

}

/* ---------- Secondary Text ---------- */

.small{

    color:#CBD5E1;

    font-size:13px;

}

/* ---------- Buttons ---------- */

.stButton>button{

    background:#2563EB;

    color:white;

    border-radius:8px;

}

.stButton>button:hover{

    background:#1D4ED8;

}

/* ---------- DataFrame ---------- */

[data-testid="stDataFrame"]{

    border-radius:10px;

}

/* ---------- Metric Widget ---------- */

[data-testid="metric-container"]{

    background:#243447;

    border-radius:10px;

    padding:10px;

}

</style>

""", unsafe_allow_html=True)

# -------------------------------------------------
# LOAD DATA
# -------------------------------------------------

priority=pd.read_csv(
f"{DATA_PATH}/data/processed/market_data_prioritised.csv"
)

reports=pd.read_csv(
f"{DATA_PATH}/reports/gpt_investigation_reports.csv"
)

# -------------------------------------------------
# SIDEBAR
# -------------------------------------------------

st.sidebar.title(" SCB AI Control Cockpit")

st.sidebar.markdown("---")

st.sidebar.success("LIVE")

st.sidebar.write(
datetime.now().strftime("%d %b %Y %H:%M:%S")
)

st.sidebar.markdown("---")

portfolio=st.sidebar.selectbox(
"Portfolio",
["All"]+sorted(priority["Portfolio"].unique().tolist())
)

risk=st.sidebar.selectbox(
"Risk Factor",
["All"]+sorted(priority["RiskFactor"].unique().tolist())
)

scenario=st.sidebar.selectbox(
"Scenario",
["All"]+sorted(priority["Scenario"].unique().tolist())
)

priority_filter=st.sidebar.selectbox(
"Priority",
["All"]+sorted(priority["FinalPriority"].unique().tolist())
)

if st.sidebar.button("🔄 Refresh"):
    st.rerun()

# -------------------------------------------------
# FILTERS
# -------------------------------------------------

df=priority.copy()

if portfolio!="All":
    df=df[df.Portfolio==portfolio]

if risk!="All":
    df=df[df.RiskFactor==risk]

if scenario!="All":
    df=df[df.Scenario==scenario]

if priority_filter!="All":
    df=df[df.FinalPriority==priority_filter]

# -------------------------------------------------
# HEADER
# -------------------------------------------------

left,right=st.columns([4,1])

with left:

    st.title("🏦 SCB AI MARKET RISK CONTROL COCKPIT")

    st.markdown(
    "<span class='live'>● LIVE</span>",
    unsafe_allow_html=True
    )

with right:

    st.metric(
        "Last Refresh",
        datetime.now().strftime("%H:%M:%S")
    )

st.divider()

# -------------------------------------------------
# KPI CALCULATIONS
# -------------------------------------------------

critical=(df.FinalPriority=="Critical").sum()

high=(df.FinalPriority=="High").sum()

medium=(df.FinalPriority=="Medium").sum()

low=(df.FinalPriority=="Low").sum()

total=len(df)

est_pnl=df.EstimatedPnL.sum()

est_var=df.EstimatedVaRImpact.sum()

feed_health=100-(df.Rule_HighLatency.mean()*100)

# -------------------------------------------------
# KPI ROW
# -------------------------------------------------

c1,c2,c3,c4,c5,c6,c7=st.columns(7)

c1.metric(
"Critical",
critical
)

c2.metric(
"High",
high
)

c3.metric(
"Medium",
medium
)

c4.metric(
"Low",
low
)

c5.metric(
"Alerts",
total
)

c6.metric(
"Est P&L",
f"${est_pnl:,.0f}"
)

c7.metric(
"Feed Health",
f"{feed_health:.1f}%"
)

st.divider()

# -------------------------------------------------
# SYSTEM STATUS
# -------------------------------------------------

st.subheader("System Status")

a,b,c,d=st.columns(4)

with a:
    st.success("Reuters Feed")

with b:
    st.success("Bloomberg Feed")

with c:

    if feed_health>95:
        st.success("Market Data Healthy")

    elif feed_health>85:
        st.warning("Minor Latency")

    else:
        st.error("Critical Feed Issues")

with d:
    st.info(f"{total} Active Alerts")

st.divider()

# -------------------------------------------------
# PLACE HOLDERS
# -------------------------------------------------

left,right=st.columns(2)

with left:

    st.subheader("📈 Alert Distribution")

    st.info("Chart coming in Part 2")

with right:

    st.subheader("📊 Business Impact")

    st.info("Chart coming in Part 2")

st.divider()

st.subheader("🚨 Live Alert Grid")

st.dataframe(
df[
[
"AlertID",
"Portfolio",
"RiskFactor",
"Scenario",
"FinalPriority",
"BusinessImpactScore",
"EstimatedPnL",
"EstimatedVaRImpact",
"ConfidenceScore"
]
].sort_values(
"BusinessImpactScore",
ascending=False
),
use_container_width=True,
height=450
)

st.divider()

st.info(
"🤖 AI Investigation Panel will be added in Part 3"
)

Overwriting app.py


In [ ]:
%%writefile -a app.py
# ============================================================
# BLOOMBERG ANALYTICS
# ============================================================

import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

plot_bg = "#0E1117"
paper_bg = "#0E1117"
font_color = "white"

# ------------------------------------------------------------
# ROW 1
# ------------------------------------------------------------

left,right = st.columns(2)

with left:

    priority_counts = (
        df["FinalPriority"]
        .value_counts()
        .reset_index()
    )

    priority_counts.columns=["Priority","Count"]

    fig = px.bar(
        priority_counts,
        x="Priority",
        y="Count",
        color="Priority",
        title="Alert Priority Distribution"
    )

    fig.update_layout(
        plot_bgcolor=plot_bg,
        paper_bgcolor=paper_bg,
        font_color=font_color,
        height=380,
        showlegend=False
    )

    st.plotly_chart(fig,use_container_width=True)

with right:

    fig = px.scatter(
        df,
        x="EstimatedVaRImpact",
        y="EstimatedPnL",
        color="FinalPriority",
        size="BusinessImpactScore",
        hover_name="RiskFactor",
        hover_data=[
            "Portfolio",
            "Scenario"
        ],
        title="Business Impact Analysis"
    )

    fig.update_layout(
        plot_bgcolor=plot_bg,
        paper_bgcolor=paper_bg,
        font_color=font_color,
        height=380
    )

    st.plotly_chart(fig,use_container_width=True)

st.divider()

# ------------------------------------------------------------
# ROW 2
# ------------------------------------------------------------

left,right = st.columns(2)

with left:

    latency = (
        df.groupby("Portfolio")["FeedLatencySeconds"]
        .mean()
        .reset_index()
    )

    fig = px.bar(
        latency,
        x="Portfolio",
        y="FeedLatencySeconds",
        color="FeedLatencySeconds",
        title="Average Feed Latency"
    )

    fig.update_layout(
        plot_bgcolor=plot_bg,
        paper_bgcolor=paper_bg,
        font_color=font_color,
        height=380
    )

    st.plotly_chart(fig,use_container_width=True)

with right:

    heat = (
        df.pivot_table(
            index="Portfolio",
            columns="FinalPriority",
            values="BusinessImpactScore",
            aggfunc="mean"
        )
        .fillna(0)
    )

    fig = px.imshow(
        heat,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale="Reds",
        title="Portfolio Risk Heatmap"
    )

    fig.update_layout(
        plot_bgcolor=plot_bg,
        paper_bgcolor=paper_bg,
        font_color=font_color,
        height=380
    )

    st.plotly_chart(fig,use_container_width=True)

st.divider()

# ------------------------------------------------------------
# ROW 3
# ------------------------------------------------------------

left,right = st.columns(2)

with left:

    scenario = (
        df.groupby("Scenario")["BusinessImpactScore"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    fig = px.bar(
        scenario,
        x="Scenario",
        y="BusinessImpactScore",
        color="BusinessImpactScore",
        title="Business Impact by Scenario"
    )

    fig.update_layout(
        plot_bgcolor=plot_bg,
        paper_bgcolor=paper_bg,
        font_color=font_color,
        height=380
    )

    st.plotly_chart(fig,use_container_width=True)

with right:

    vendor = (
        df["VendorAgreement"]
        .value_counts()
        .reset_index()
    )

    vendor.columns=["Agreement","Count"]

    fig = px.pie(
        vendor,
        names="Agreement",
        values="Count",
        title="Vendor Agreement"
    )

    fig.update_layout(
        plot_bgcolor=plot_bg,
        paper_bgcolor=paper_bg,
        font_color=font_color,
        height=380
    )

    st.plotly_chart(fig,use_container_width=True)

st.divider()

# ------------------------------------------------------------
# ROW 4
# ------------------------------------------------------------

timeline = df.copy()

timeline["Timestamp"] = pd.to_datetime(
    timeline["Timestamp"]
)

timeline = (
    timeline
    .groupby(
        timeline["Timestamp"].dt.floor("H")
    )
    .size()
    .reset_index(name="Alerts")
)

fig = px.line(
    timeline,
    x="Timestamp",
    y="Alerts",
    markers=True,
    title="Alert Timeline"
)

fig.update_layout(
    plot_bgcolor=plot_bg,
    paper_bgcolor=paper_bg,
    font_color=font_color,
    height=420
)

st.plotly_chart(
    fig,
    use_container_width=True
)

st.divider()

Appending to app.py


In [ ]:
%%writefile -a app.py
# ============================================================
# PART 3 - ALERT EXPLORER & AI INVESTIGATION
# ============================================================

st.header("🚨 Alert Explorer")

search = st.text_input(
    "Search Risk Factor / Portfolio / Alert ID",
    ""
)

table = df.copy()

if search:

    table = table[
        table["RiskFactor"].astype(str).str.contains(search, case=False) |
        table["Portfolio"].astype(str).str.contains(search, case=False) |
        table["AlertID"].astype(str).str.contains(search, case=False)
    ]

table = table.sort_values(
    "BusinessImpactScore",
    ascending=False
)

display_cols = [
    "AlertID",
    "Portfolio",
    "RiskFactor",
    "Scenario",
    "FinalPriority",
    "BusinessImpactScore",
    "EstimatedPnL",
    "EstimatedVaRImpact",
    "ConfidenceScore"
]

display_cols = [c for c in display_cols if c in table.columns]

st.dataframe(
    table[display_cols],
    use_container_width=True,
    height=300
)

st.divider()

# ============================================================
# ALERT SELECTION
# ============================================================

alert_list = table["AlertID"].unique().tolist()

selected_alert = st.selectbox(
    "Select Alert",
    alert_list
)

selected = table[
    table.AlertID == selected_alert
].iloc[0]

# ============================================================
# SUMMARY
# ============================================================

st.header("📌 Alert Summary")

c1,c2,c3,c4 = st.columns(4)

c1.metric(
    "Priority",
    selected["FinalPriority"]
)

c2.metric(
    "Business Score",
    round(selected["BusinessImpactScore"],1)
)

c3.metric(
    "Estimated P&L",
    f"${selected['EstimatedPnL']:,.0f}"
)

c4.metric(
    "Estimated VaR",
    f"${selected['EstimatedVaRImpact']:,.0f}"
)

st.divider()

# ============================================================
# DETAILS
# ============================================================

left,right = st.columns([1,1])

with left:

    st.subheader("Market Data")

    st.write("**Risk Factor:**", selected["RiskFactor"])
    st.write("**Portfolio:**", selected["Portfolio"])
    st.write("**Scenario:**", selected["Scenario"])

    st.write("**Reuters:**", round(selected["ReutersPrice"],4))
    st.write("**Bloomberg:**", round(selected["BloombergPrice"],4))

    st.write("**Feed Latency:**",
             f"{selected['FeedLatencySeconds']} sec")

    st.write("**Vendor Agreement:**",
             selected["VendorAgreement"])

    st.write("**Market Wide Move:**",
             selected["MarketWideMove"])

with right:

    st.subheader("🛡️ Deterministic Rule Engine")

    rules = [

        ("Stale Price", selected["Rule_Stale"]),

        ("Large Price Move", selected["Rule_LargeMove"]),

        ("Z-Score Outlier", selected["Rule_ZScoreOutlier"]),

        ("Source Divergence", selected["Rule_SourceDivergence"]),

        ("Feed Latency", selected["Rule_HighLatency"])

    ]

    for rule_name, triggered in rules:

        if triggered:

            st.error(f"🚨 {rule_name}")

        else:

            st.success(f"✔ {rule_name}")

    st.metric(

        "Triggered Rules",

        int(selected["TriggeredRuleCount"])

    )

    st.metric(

        "Rule Severity",

        selected["RuleSeverity"]

    )

    st.metric(

        "Rule Score",

        round(selected["RuleSeverityScore"],2)

    )

    if "RuleEvidence" in selected.index:

        st.markdown("**Evidence**")

        st.info(selected["RuleEvidence"])

st.divider()

st.header("🤖 AI Assessment")

left,right = st.columns(2)

with left:

    st.metric(

        "AI Severity",

        selected["AISeverity"]

    )

    st.metric(

        "Confidence",

        f"{selected['ConfidenceScore']}%"

    )

with right:

    st.markdown("### Recommendation")

    st.info(selected["Recommendation"])

    st.markdown("### Business Recommendation")

    st.success(selected["BusinessRecommendation"])

# ============================================================
# GPT REPORT
# ============================================================

st.header("🤖 AI Investigation Report")

report_col = None

for col in reports.columns:

    if "report" in col.lower():
        report_col = col
        break

if report_col is None:

    st.warning("No GPT report column found.")

else:

    report = reports[
        reports.AlertID == selected_alert
    ]

    if len(report):

        st.markdown(
            report.iloc[0][report_col]
        )

    else:

        st.warning(
            "GPT report not found for selected alert."
        )

st.divider()

# ============================================================
# ROOT CAUSE ANALYTICS
# ============================================================

st.header("📊 Root Cause Analytics")

left,right = st.columns(2)

with left:

    root = (
        df["MostCommonRootCause"]
        .value_counts()
        .reset_index()
    )

    root.columns = ["Root Cause","Count"]

    fig = px.bar(
        root,
        x="Root Cause",
        y="Count",
        color="Count",
        title="Historical Root Causes"
    )

    fig.update_layout(
        plot_bgcolor="#0E1117",
        paper_bgcolor="#0E1117",
        font_color="white"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

with right:

    fig = px.scatter(
        df,
        x="ConfidenceScore",
        y="BusinessImpactScore",
        color="FinalPriority",
        size="EstimatedPnL",
        hover_name="RiskFactor",
        title="AI Confidence vs Business Impact"
    )

    fig.update_layout(
        plot_bgcolor="#0E1117",
        paper_bgcolor="#0E1117",
        font_color="white"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

st.divider()

st.success("✅ AI Investigation Completed")

Appending to app.py


In [ ]:
!pkill -f streamlit || true
!pkill -f ngrok || true

^C
^C


In [ ]:
!streamlit run app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.headless true \
  >/content/streamlit.log 2>&1 &

In [ ]:
import time
time.sleep(5)
print("Streamlit startup attempted")

Streamlit startup attempted


In [ ]:
!ps aux | grep streamlit

root       21257 40.2  0.1 523328 69912 ?        Sl   02:06   0:02 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true
root       21332  0.0  0.0   7372  3396 ?        S    02:06   0:00 /bin/bash -c ps aux | grep streamlit
root       21334  0.0  0.0   6480  2468 ?        S    02:06   0:00 grep streamlit


In [ ]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Sun, 19 Jul 2026 02:06:45 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6602
last-modified: Sun, 19 Jul 2026 01:06:43 GMT
etag: "84170e9fdd5d586c87865b4d077b9745"
cache-control: no-cache



In [ ]:
!tail -100 /content/streamlit.log



2026-07-19 02:06:42.628 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.199.59.109:8501



In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
print(ngrok.get_tunnels())

[]


In [ ]:
public_url = ngrok.connect(
    8501,
    "http"
)

print(public_url.public_url)

https://f32f-35-199-59-109.ngrok-free.app
